# Genomics EDA diagnostics

This notebook stays helper-backed on purpose: the reusable logic lives in `jaguar_geo_assign.reporting.genomics_diagnostics`, and the notebook now exercises a larger realistic sample while emitting a JSON payload under `reports/generated/`.

In [ ]:
from pathlib import Path
from pprint import pprint

from jaguar_geo_assign.reporting import build_eda_payload, write_eda_payload_json


def build_realistic_records(*, total: int, source: str) -> list[dict[str, object]]:
    records: list[dict[str, object]] = []
    for index in range(total):
        length = 32 if index % 2 == 0 else 36
        reference_sequence = ('ACGT' * ((length // 4) + 1))[:length]
        sequence = reference_sequence
        if source == 'consensus':
            pattern = index % 4
            if pattern == 2:
                sequence = 'T' + reference_sequence[1:]
            elif pattern == 3:
                sequence = reference_sequence[:-1] + 'N'
        records.append(
            {
                'sample_id': f'{source}-{index}',
                'locus_id': f'chr{1 + (index % 3)}:block-{index}',
                'split': 'train' if index % 5 else 'validation',
                'source': source,
                'sequence': sequence,
                'reference_sequence': reference_sequence,
                'variant_count': 0 if sequence == reference_sequence else 1,
                'callable_bases': len(sequence) - sequence.count('N'),
                'filtered_bases': 0,
                'no_call_bases': sequence.count('N'),
                'token_count': max(1, len(sequence) // 4),
            }
        )
    return records


consensus_records = build_realistic_records(total=96, source='consensus')
baseline_records = build_realistic_records(total=48, source='reference')
report_path = Path('reports/generated/feline_pretrain/diagnostics_sample_payload.json')
payload = build_eda_payload(consensus_records, baseline_records, near_duplicate_sample_limit=64)
write_eda_payload_json(
    consensus_records,
    baseline_records,
    report_path,
    near_duplicate_sample_limit=64,
)
pprint(payload['consensus_sample_overview'])
pprint(payload['consensus_corpus'])
pprint(payload['baseline_comparison'])
print(f'Report written to {report_path}')
